# Figure S7 -- the matching, by example

Panels (a) and (b) show the two steps of the matching on one region, one cycle and
one lead day: every forecast eddy joined to the nearest observed eddy of the same
polarity, then each contest resolved in favour of the closest pair. The separation
cap is applied after that resolution, not as part of it, so pairs beyond it are
drawn dashed rather than removed. Panel (c) is the distribution of the resulting
one-to-one separations on lead day 1, pooled over every cycle and region, which is
what justifies the threshold from the data.

**What this reproduces.** Panel (c) in full. For (a) and (b), a link has a forecast
end and an observed end; whether the observed centres are in the deposit decides
what can be drawn, and the notebook reports which case it is in:

* **observed centres present** -- the panels are drawn as published, without the
  altimetry field that shades the background.
* **absent** -- each link still carries its separation and its bearing from the
  forecast centre, so the offsets are drawn as arrows from the forecast eddies
  instead of as segments between the two sets.


In [ ]:
from pathlib import Path
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

# Folder holding the archived NetCDF files. Default: alongside this notebook.
DATA = Path(".")

FIGDIR = Path("figures")
FIGDIR.mkdir(exist_ok=True)

REGION_ORDER = ["Kuroshio", "Gulf of Mexico", "Arabian Sea"]


def by_region(paths):
    """Open each file, read its region attribute, return them in paper order."""
    tagged = []
    for p in paths:
        with xr.open_dataset(p) as ds:
            tagged.append((ds.attrs.get("region", p.stem), p))
    key = lambda t: (REGION_ORDER.index(t[0]) if t[0] in REGION_ORDER else 99, t[0])
    return sorted(tagged, key=key)


def need(pattern):
    found = sorted(DATA.glob(pattern))
    if not found:
        raise FileNotFoundError(
            f"No file matching {pattern!r} in {DATA.resolve()}. "
            f"Set DATA to the folder holding the archived NetCDF files.")
    return found


In [ ]:
try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    PROJ, HAVE_CARTOPY = ccrs.PlateCarree(), True
except ImportError:
    PROJ, HAVE_CARTOPY = None, False
    print("cartopy is not installed -- drawing the maps without coastlines")

# Cartopy fetches its coastline data from the network the first time it is used.
# Try once here, so a machine that is offline or has no cached copy degrades to a
# map without coastlines instead of failing in the middle of a figure.
HAVE_COAST = False
if HAVE_CARTOPY:
    try:
        next(iter(cfeature.COASTLINE.geometries()))
        HAVE_COAST = True
    except Exception as exc:
        print(f"cartopy cannot reach its coastline data ({type(exc).__name__}) -- "
              f"drawing the maps without coastlines")

TKW = dict(transform=PROJ) if HAVE_CARTOPY else {}


def map_features(ax, labels=True, land="lightgray", lw=0.4, zland=3, zcoast=4):
    """Coastlines and gridlines where available; plain axes otherwise."""
    if HAVE_COAST:
        ax.add_feature(cfeature.LAND, facecolor=land, zorder=zland)
        ax.add_feature(cfeature.COASTLINE, linewidth=lw, zorder=zcoast)
    if HAVE_CARTOPY:
        gl = ax.gridlines(draw_labels=labels, linestyle="--", lw=0.2)
        gl.top_labels = gl.right_labels = False
        return gl
    ax.tick_params(labelsize=7)
    return None
POL_LINE  = {1: "#C0392B", -1: "#1B6CA8"}
FAR_COLOR = "0.45"
SSH_LIM   = 0.4

path = need("FigureS7_matching_example_*.nc")[0]
ab = xr.open_dataset(path)
MAX_DIST_KM = float(ab.attrs["maximum_separation_km"])
HAVE_OBS = "observed_lon" in ab
print("observed centres in the deposit:", HAVE_OBS)

R_EARTH = 6371.0


def endpoint(lon, lat, dist_km, bearing_deg):
    """Where a link of this length and bearing ends, on a sphere."""
    d = dist_km / R_EARTH
    b = np.deg2rad(bearing_deg)
    p1, l1 = np.deg2rad(lat), np.deg2rad(lon)
    p2 = np.arcsin(np.sin(p1) * np.cos(d) + np.cos(p1) * np.sin(d) * np.cos(b))
    l2 = l1 + np.arctan2(np.sin(b) * np.sin(d) * np.cos(p1),
                         np.cos(d) - np.sin(p1) * np.sin(p2))
    return np.rad2deg(l2), np.rad2deg(p2)


def draw_step(ax, tag, show_cap, box_contested):
    sep = ab[f"{tag}_separation"].values
    brg = ab[f"{tag}_bearing"].values
    pol = ab[f"{tag}_polarity"].values
    fi  = ab[f"{tag}_forecast_index"].values
    flon = ab.forecast_lon.values[fi]
    flat = ab.forecast_lat.values[fi]
    if HAVE_OBS:
        oi = ab[f"{tag}_observed_index"].values
        olon, olat = ab.observed_lon.values[oi], ab.observed_lat.values[oi]
    else:
        olon, olat = endpoint(flon, flat, sep, brg)
    for k in range(len(sep)):
        far = show_cap and sep[k] > MAX_DIST_KM
        ax.plot([flon[k], olon[k]], [flat[k], olat[k]],
                color=FAR_COLOR if far else POL_LINE[int(pol[k])],
                lw=0.9 if far else 1.1,
                ls=(0, (2.2, 1.6)) if far else "-", zorder=6, **TKW)
    if box_contested and "step1_contested" in ab:
        m = ab.step1_contested.values == 1
        ax.plot(olon[m], olat[m], marker="s", ms=9, mfc="none", mec="k",
                mew=0.9, ls="none", zorder=8, **TKW)
    if HAVE_OBS:
        ax.plot(ab.observed_lon, ab.observed_lat, "o", ms=2.6, mfc="white",
                mec="k", mew=0.5, ls="none", zorder=7, **TKW)
    ax.plot(ab.forecast_lon, ab.forecast_lat, "o", ms=2.6, mfc="k", mec="k",
            ls="none", zorder=7, **TKW)


In [ ]:
fig = plt.figure(figsize=(13, 5.2), dpi=130)
gs = fig.add_gridspec(2, 2, width_ratios=[1.25, 1.0], hspace=0.30, wspace=0.22)

for r, (tag, title, cap, boxit) in enumerate(
        [("step1", "After step 1", False, True),
         ("step2", "After step 2", True,  False)]):
    ax = (fig.add_subplot(gs[r, 0], projection=PROJ) if HAVE_CARTOPY
          else fig.add_subplot(gs[r, 0]))
    pcm = ax.pcolormesh(ab.lon, ab.lat, ab.ssh, cmap="RdYlBu_r",
                        vmin=-SSH_LIM, vmax=SSH_LIM, alpha=0.9, shading="auto",
                        zorder=1, **TKW)
    ax.contour(ab.lon, ab.lat, ab.ssh,
               levels=np.arange(-SSH_LIM, SSH_LIM + 0.01, 0.1),
               colors="0.25", linewidths=0.35, zorder=2, **TKW)
    draw_step(ax, tag, cap, boxit)
    gl = map_features(ax, lw=0.3, zland=4, zcoast=5)
    if gl is not None:
        gl.xlabel_style = gl.ylabel_style = {"size": 6.5}
    ax.set_title(title, fontsize=9, fontweight="bold")
    ax.text(0.015, 0.95, f"({chr(97 + r)})", transform=ax.transAxes,
            fontsize=9, fontweight="bold", va="top", zorder=12,
            bbox=dict(fc="white", alpha=0.85, pad=0.8, ec="none"))

# ---- (c) the separation distribution -------------------------------------
pc = need("FigureS7_separations_leadday*.nc")[0]
with xr.open_dataset(pc) as cds:
    s = cds.separation.values
    thr = float(cds.attrs["matching_threshold_km"])
    ax = fig.add_subplot(gs[:, 1])
    bins = np.arange(0, 480 + 1, 10)
    ax.hist(s, bins=bins, color="0.72", edgecolor="0.35", lw=0.4,
            weights=100 * np.ones_like(s) / s.size)
    ax.set_xlabel("Separation (km)", fontsize=9)
    ax.set_ylabel("Percent", fontsize=9)
    ax.set_xlim(0, bins[-1])
    ax.tick_params(labelsize=7)
    ax2 = ax.twinx()
    ss = np.sort(s)
    ax2.plot(ss, 100 * np.arange(1, ss.size + 1) / ss.size, "k--", lw=1.1)
    ax2.set_ylim(0, 100)
    ax2.set_ylabel("Cumulative (%)", fontsize=9)
    ax2.tick_params(labelsize=7)
    ax2.axvline(thr, color="0.25", lw=1.2)
    inside = 100 * np.mean(s <= thr)
    ax2.annotate(f"{inside:.0f}% within\n{thr:g} km", xy=(thr, inside),
                 xytext=(-8, -26), textcoords="offset points", ha="right",
                 fontsize=7, color="0.25")
    ax.text(0.03, 0.96, "(c)", transform=ax.transAxes, fontsize=9,
            fontweight="bold", va="top")

out = FIGDIR / "FigureS7_matching_example.png"
fig.savefig(out, dpi=300, bbox_inches="tight")
plt.show()
print("wrote", out)
print(f"  {inside:.1f}% within {thr:g} km, median {np.median(s):.1f} km, "
      f"n = {s.size}")
ab.close()
